# E8-v2 — Colab CPU validation and execution

This notebook runs the **pretrained FastText Common Crawl `cc.en.300.bin`** retrieval variant. It does not train or fine-tune FastText. The original E8 experiment is not modified.

**Important:** the full binary is multi-GB. The notebook checks free disk and RAM before download. It streams the official gzip artifact to temporary runtime storage, loads it once, and does not place it in Git or Google Drive. The model remains in memory during retrieval.

ToolRet and MCPToolBench++ are evaluated as separate tracks. Primary metrics remain Hit@1, Hit@5, MRR@10 and Macro Hit@1.

**Dataset compatibility:** this notebook now reuses the dataset layout from the previous MultiBenchmark Colab run under `/content/taas-provenance-discovery/data`. It does not require a Google Drive dataset copy. Colab runtimes are ephemeral: if the previous runtime was reset, rerun the original pinned acquisition/preprocessing workflow before this benchmark.

In [ ]:
import os, sys, json, shutil, subprocess, pathlib, platform, time, gc
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU cores:', os.cpu_count())
print('Disk free GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))
try:
    import psutil
    vm=psutil.virtual_memory()
    print('RAM total GiB:', round(vm.total/1024**3,2), '| available GiB:', round(vm.available/1024**3,2))
except ImportError:
    print('psutil not installed yet')


## 1. Clone the experiment branch and reuse the previous Colab dataset layout

The previous MultiBenchmark workflow stored datasets inside the repository checkout:
- ToolRet: `data/processed/toolret`
- MCPToolBench++: `data/raw`

The notebook checks out the exact E8-v2 branch first, then validates these paths. It deliberately does not silently download or rebuild benchmark data.

In [ ]:
REPO_URL = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
BRANCH = 'feature/e8-cpu-fasttext-v2'
REPO_DIR = Path('/content/taas-provenance-discovery')

# Reuse an existing checkout without deleting untracked dataset files.
if not REPO_DIR.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO_DIR)],check=True)
else:
    origin = subprocess.check_output(
        ['git','-C',str(REPO_DIR),'remote','get-url','origin'],text=True
    ).strip().rstrip('/')
    if origin not in {REPO_URL, REPO_URL.removesuffix('.git')}:
        raise RuntimeError(f'Unexpected repository origin at {REPO_DIR}: {origin}')
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','--depth','1','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','-B',BRANCH,'FETCH_HEAD'],check=True)

TOOLRET_DIR = REPO_DIR / 'data/processed/toolret'
MCP_RAW_DIR = REPO_DIR / 'data/raw'
RESULTS_DIR = Path('/content/e8-results/E8-CPU-StaticEmbedding-v2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Do not overwrite a valid existing adapter. Missing ToolRet files are rebuilt
# from the official, revision-pinned Hugging Face datasets in the next cell.
print('Repository:', REPO_DIR)
print('ToolRet adapter present:', (TOOLRET_DIR/'manifest.json').is_file())
print('MCP raw data present:', (MCP_RAW_DIR/'SOURCES.txt').is_file())

## 2. Install CPU dependencies in the checked-out experiment branch

The notebook uses the existing checkout and never deletes it, because it contains the datasets acquired by the previous Colab workflow. No benchmark dataset or model artifact is committed.

In [ ]:
%cd /content/taas-provenance-discovery
import sys, subprocess

def run_install(args, label):
    print(f"\n=== {label} ===", flush=True)
    result = subprocess.run(args, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} failed (exit={result.returncode}). "
            "The full pip/compiler diagnostic is shown above; do not continue to dataset acquisition."
        )

# Install the project and ordinary Python dependencies independently of FastText.
run_install(
    [sys.executable, "-m", "pip", "install", "-e", ".", "pytest", "datasets==3.3.2"],
    "Project, pytest and Hugging Face datasets installation"
)

# fasttext-wheel 0.9.2 publishes Linux wheels through CPython 3.12, but no
# CPython 3.13 wheel. On 3.13, build the same pinned release from its source
# distribution using a modern pybind11; never silently substitute another
# FastText implementation or version.
if sys.version_info >= (3, 13):
    run_install(
        [sys.executable, "-m", "pip", "install", "setuptools>=68", "wheel", "pybind11>=2.13.6"],
        "CPython 3.13 FastText build prerequisites"
    )
    run_install(
        [sys.executable, "-m", "pip", "install", "--no-build-isolation",
         "--no-binary=fasttext-wheel", "fasttext-wheel==0.9.2"],
        "Build pinned fasttext-wheel 0.9.2 from source for CPython 3.13"
    )
else:
    run_install(
        [sys.executable, "-m", "pip", "install", "--only-binary=:all:", "fasttext-wheel==0.9.2"],
        "Install pinned fasttext-wheel 0.9.2 wheel"
    )

# Fail early with a useful diagnostic if the native extension is not importable.
import fasttext
print("FastText import:", fasttext.__file__)
print("FastText version:", __import__("importlib.metadata").metadata.version("fasttext-wheel"))
print("Python:", sys.version)
print("Checked out:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())


In [ ]:
## ToolRet dataset acquisition and deterministic adapter build
# Source repositories and revisions are pinned to the earlier acquisition protocol.
# This downloads only the official ToolRet benchmark data (not model weights).
import hashlib, json, math
from datasets import load_dataset
from pathlib import Path

TOOLRET_QUERIES_REPO = "mangopy/ToolRet-Queries"
TOOLRET_TOOLS_REPO = "mangopy/ToolRet-Tools"
TOOLRET_QUERIES_REV = "b8c76ad3349ff17497b6bdb28bb5b8f61a0f6445"
TOOLRET_TOOLS_REV = "e06c38c75612b6536bd959e08cdd345894aba6a7"

TASK_CONFIGS = {
    "apibank":"web", "apigen":"web", "mnms":"web", "reversechain":"web",
    "rotbench":"web", "t-eval-dialog":"web", "t-eval-step":"web",
    "taskbench-daily":"web", "toolace":"web", "toolbench":"web",
    "toolemu":"web", "tooleyes":"web", "ultratool":"web",
    "autotools-food":"web", "autotools-music":"web", "autotools-weather":"web",
    "restgpt-spotify":"web", "restgpt-tmdb":"web",
    "craft-math-algebra":"code", "craft-tabmwp":"code", "craft-vqa":"code",
    "gorilla-huggingface":"code", "gorilla-pytorch":"code", "gorilla-tensor":"code",
    "toolink":"code",
    "appbench":"customized", "gpt4tools":"customized", "gta":"customized",
    "taskbench-huggingface":"customized", "taskbench-multimedia":"customized",
    "metatool":"customized", "tool-be-honest":"customized",
    "toolalpaca":"customized", "toolbench-sam":"customized",
}

def _json_value(value, label):
    if isinstance(value, (dict, list)):
        return value
    try:
        return json.loads(value)
    except Exception as exc:
        raise ValueError(f"Cannot parse official ToolRet {label} JSON: {str(value)[:180]}") from exc

def _sha256_json(path):
    h=hashlib.sha256()
    with open(path,"rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

def build_toolret_adapter():
    required = [
        TOOLRET_DIR/"manifest.json",
        TOOLRET_DIR/"discovery"/"tools.json",
        TOOLRET_DIR/"discovery"/"tasks.json",
        TOOLRET_DIR/"evaluator"/"task_references.json",
    ]
    if all(p.is_file() for p in required):
        print("Existing complete ToolRet adapter found; preserving it.")
        return

    print("Downloading pinned official ToolRet tool corpus...")
    tool_rows=[]
    for category in ("code","customized","web"):
        ds=load_dataset(TOOLRET_TOOLS_REPO,category,revision=TOOLRET_TOOLS_REV,split="tools")
        if set(ds.column_names) != {"id","documentation"}:
            raise ValueError(f"Unexpected ToolRet tools schema for {category}: {ds.column_names}")
        for row in ds:
            tool_id=str(row["id"]).strip()
            raw_doc=str(row["documentation"] or "").strip()
            if not tool_id or not raw_doc:
                raise ValueError(f"Empty ToolRet tool ID/documentation in {category}")
            try:
                doc=json.loads(raw_doc)
            except Exception:
                doc={}
            if not isinstance(doc,dict):
                doc={}
            tool_rows.append({
                "tool_id":tool_id,
                "name":str(doc.get("name") or tool_id),
                # Keep the complete original documentation to avoid dropping fields.
                "description":raw_doc,
                "input_schema":doc.get("parameters") if isinstance(doc.get("parameters"),dict) else {},
                "category":category,
                "server_id":"toolret_"+category,
            })

    print("Downloading pinned official ToolRet query configs...")
    task_rows=[]
    qrel_rows=[]
    for config,category in TASK_CONFIGS.items():
        ds=load_dataset(TOOLRET_QUERIES_REPO,config,revision=TOOLRET_QUERIES_REV,split="queries")
        expected={"id","query","instruction","labels","category"}
        if set(ds.column_names) != expected:
            raise ValueError(f"Unexpected ToolRet query schema for {config}: {ds.column_names}")
        for row in ds:
            task_id=str(row["id"]).strip()
            query=str(row["query"] or "").strip()
            if not task_id or not query:
                raise ValueError(f"Empty ToolRet task ID/query in {config}")
            labels=_json_value(row["labels"],f"labels for {task_id}")
            if not isinstance(labels,list):
                raise ValueError(f"ToolRet labels must be a list for {task_id}")
            task_rows.append({
                "task_id":task_id,
                "prompt":query,
                "category":category,
            })
            judgments=[]
            for label in labels:
                if not isinstance(label,dict) or not label.get("id"):
                    raise ValueError(f"Malformed ToolRet relevance label for {task_id}: {label}")
                judgments.append({
                    "tool_id":str(label["id"]),
                    "relevance":float(label.get("relevance",1)),
                })
            qrel_rows.append({"task_id":task_id,"relevance_judgments":judgments})

    tool_ids=[r["tool_id"] for r in tool_rows]
    task_ids=[r["task_id"] for r in task_rows]
    if len(tool_ids)!=len(set(tool_ids)):
        raise ValueError("Duplicate tool IDs in pinned ToolRet corpus; refusing to write adapter.")
    if len(task_ids)!=len(set(task_ids)):
        raise ValueError("Duplicate task IDs in pinned ToolRet queries; refusing to write adapter.")
    tool_id_set=set(tool_ids)
    unresolved=sorted({j["tool_id"] for row in qrel_rows for j in row["relevance_judgments"]}-tool_id_set)
    if unresolved:
        raise ValueError(f"{len(unresolved)} ToolRet qrel IDs are absent from the complete candidate corpus; first: {unresolved[:10]}")

    # Integrity sentinels from the previously processed ToolRet corpus.
    if len(tool_rows)!=44453 or len(task_rows)!=7960:
        raise ValueError(
            f"Unexpected pinned ToolRet cardinality: tools={len(tool_rows)} tasks={len(task_rows)}; "
            "expected 44,453 tools and 7,960 tasks. No adapter written."
        )

    staging=TOOLRET_DIR.with_name("toolret.building")
    if staging.exists():
        import shutil
        shutil.rmtree(staging)
    (staging/"discovery").mkdir(parents=True)
    (staging/"evaluator").mkdir(parents=True)
    (staging/"discovery"/"tools.json").write_text(json.dumps(tool_rows,ensure_ascii=False))
    (staging/"discovery"/"tasks.json").write_text(json.dumps(task_rows,ensure_ascii=False))
    (staging/"evaluator"/"task_references.json").write_text(json.dumps(qrel_rows,ensure_ascii=False))
    manifest={
        "dataset":"ToolRet",
        "queries_repo":TOOLRET_QUERIES_REPO,"queries_revision":TOOLRET_QUERIES_REV,
        "tools_repo":TOOLRET_TOOLS_REPO,"tools_revision":TOOLRET_TOOLS_REV,
        "task_configs":TASK_CONFIGS,
        "task_count":len(task_rows),"candidate_tool_count":len(tool_rows),
        "qrel_task_count":sum(bool(r["relevance_judgments"]) for r in qrel_rows),
        "qrel_rows":len(qrel_rows),
        "unresolved_qrel_tool_ids":0,
        "construction":"query.prompt=query; official labels retained only under evaluator; complete 3-category tool corpus used as candidates",
    }
    (staging/"manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+"\n")
    import shutil
    if TOOLRET_DIR.exists():
        shutil.rmtree(TOOLRET_DIR)
    staging.rename(TOOLRET_DIR)
    print(json.dumps(manifest,indent=2))

build_toolret_adapter()

# Validate all expected paths before model download.
required_toolret = [
    TOOLRET_DIR/"manifest.json",
    TOOLRET_DIR/"discovery"/"tools.json",
    TOOLRET_DIR/"discovery"/"tasks.json",
    TOOLRET_DIR/"evaluator"/"task_references.json",
]
required_mcp = [MCP_RAW_DIR/"SOURCES.txt", MCP_RAW_DIR/"mcptoolbenchpp"]
missing=[str(p) for p in required_toolret+required_mcp if not p.exists()]
if missing:
    raise FileNotFoundError("Dataset inputs still missing:\\n- "+"\\n- ".join(missing))
print("ToolRet and MCPToolBench++ dataset paths validated.")

## 3. Run unit tests and syntax/config checks

These checks validate the implementation mechanics only; they do not establish retrieval quality or prove that the real pretrained model loads successfully.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_e8_cpu_static_embedding.py'],check=True)
subprocess.run([sys.executable,'-m','py_compile','experiments/e8_cpu_static_embedding.py'],check=True)
subprocess.run([sys.executable,'-m','json.tool','docs/experiment_design/E8_CPU_STATIC_EMBEDDING_v2.json'],stdout=subprocess.DEVNULL,check=True)
print('Unit tests, Python compilation and JSON validation passed.')

## 4. Resource gate and load pretrained model (inference only)

The official Common Crawl model is large. The 8 GiB free-disk threshold is a safety gate, not a guarantee that RAM is sufficient. If available RAM is below 10 GiB, stop and select a larger Colab runtime rather than risking an OOM. Do not switch to a different model silently.

In [ ]:
import psutil
free_gib = shutil.disk_usage('/content').free / 1024**3
available_ram_gib = psutil.virtual_memory().available / 1024**3
print(f'Free disk: {free_gib:.2f} GiB | Available RAM: {available_ram_gib:.2f} GiB')
if free_gib < 8.0:
    raise RuntimeError('STOP: less than 8 GiB free disk for streamed model download.')
if available_ram_gib < 10.0:
    raise RuntimeError('STOP: less than 10 GiB available RAM. Choose a higher-RAM CPU runtime.')

from experiments.e8_cpu_static_embedding import prepare_pretrained_model
load_started = time.perf_counter()
fasttext_model, model_sha256, model_metadata = prepare_pretrained_model(
    model_path=None, cache_dir=None, min_free_gb=8.0
)
load_seconds = time.perf_counter() - load_started
print(json.dumps({'model':model_metadata,'sha256':model_sha256,'load_seconds':load_seconds,
                   'available_ram_after_load_gib':round(psutil.virtual_memory().available/1024**3,2)},indent=2))
assert fasttext_model.get_dimension() == 300
probe = fasttext_model.get_word_vector('unseen_mcp_tool_identifier_987654')
assert len(probe) == 300 and all(map(lambda x: float(x)==float(x), probe))
print('300d model and subword/OOV vector probe passed.')

## 5. Load both benchmark tracks and inspect denominators

The evaluator labels are consumed only by the metric evaluator after rankings have been generated. The IDF model is fitted only over candidate tool documents, independently per track.

In [ ]:
from experiments.e8_retrieval_campaign import load_toolret_track, load_mcptoolbenchpp_track
toolret = load_toolret_track(TOOLRET_DIR)
mcpt = load_mcptoolbenchpp_track(MCP_RAW_DIR)
for t in (toolret, mcpt):
    print(json.dumps({'track':t.dataset_id,'tasks':len(t.tasks),'candidate_tools':len(t.tools),
                      'qrel_tasks':sum(bool(t.qrels.get(x.task_id)) for x in t.tasks),
                      'exclusions':t.exclusions},indent=2))

## 6. Run 100-query smoke tests independently

Run this gate first. It uses the complete candidate tool universe but only the first 100 discovery queries per track. It writes smoke-test artifacts to Drive, separate from the full-run directory.

In [ ]:
from dataclasses import replace
from experiments.e8_cpu_static_embedding import run_track
SMOKE_DIR = RESULTS_DIR/'smoke_100'
SMOKE_DIR.mkdir(parents=True,exist_ok=True)
def first_n_tasks(track, n=100):
    tasks=track.tasks[:n]
    ids={t.task_id for t in tasks}
    return replace(track, tasks=tasks,
                   qrels={k:v for k,v in track.qrels.items() if k in ids},
                   cluster_by_task={k:v for k,v in track.cluster_by_task.items() if k in ids},
                   secondary_qrels={k:v for k,v in track.secondary_qrels.items() if k in ids})

smoke_summaries=[]
for track in (toolret, mcpt):
    smoke=first_n_tasks(track,100)
    out=SMOKE_DIR/track.dataset_id
    result=run_track(smoke,fasttext_model,out,model_sha256,{'top_k':10,'checkpoint_tasks':16})
    summary={'track':track.dataset_id,'smoke_tasks':len(smoke.tasks),'candidate_tools':len(smoke.tools),
             'primary_metrics':result['metrics_primary_qrels']}
    smoke_summaries.append(summary)
    print(json.dumps(summary,indent=2))
(SMOKE_DIR/'SMOKE_SUMMARY.json').write_text(json.dumps(smoke_summaries,indent=2)+'\n')

## 7. Full benchmark — run only after reviewing smoke results

Set `RUN_FULL = True` only after checking the smoke outputs, memory headroom, dataset manifests and expected runtime. This runs each dataset track separately and saves rankings/checkpoints/metrics to Drive.

In [ ]:
RUN_FULL = False  # Explicit opt-in after smoke-test review
if RUN_FULL:
    FULL_DIR = RESULTS_DIR/'full'
    FULL_DIR.mkdir(parents=True,exist_ok=True)
    full_summaries=[]
    for track in (toolret, mcpt):
        out=FULL_DIR/track.dataset_id
        result=run_track(track,fasttext_model,out,model_sha256,{'top_k':10,'checkpoint_tasks':16})
        summary={'track':track.dataset_id,'tasks':len(track.tasks),'candidate_tools':len(track.tools),
                 'primary_metrics':result['metrics_primary_qrels'],
                 'secondary_sensitivity_metrics':result['metrics_secondary_sensitivity_qrels'],
                 'timing_seconds':result['timing_seconds']}
        full_summaries.append(summary)
        print(json.dumps(summary,indent=2))
    (FULL_DIR/'FULL_SUMMARY.json').write_text(json.dumps(full_summaries,indent=2)+'\n')
else:
    print('Full run is disabled. Set RUN_FULL=True only after smoke validation.')


## 8. Export a compact validation record

This record includes the exact Git commit, model hash, runtime telemetry, smoke results and full-run summary if executed. It does not include the model binary.

In [ ]:
validation_record={
 'experiment':'E8-CPU-StaticEmbedding-v2','training_performed':False,
 'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),
 'branch':BRANCH,'model_sha256':model_sha256,'model_metadata':model_metadata,
 'model_load_seconds':load_seconds,'runtime':{'python':sys.version,'platform':platform.platform(),
             'cpu_count':os.cpu_count(),'ram_available_gib_after_load':round(psutil.virtual_memory().available/1024**3,2)},
 'smoke_summary':smoke_summaries,'full_run_enabled':RUN_FULL}
(RESULTS_DIR/'COLAB_VALIDATION_RECORD.json').write_text(json.dumps(validation_record,indent=2)+'\n')
print('Validation record:', RESULTS_DIR/'COLAB_VALIDATION_RECORD.json')
print(json.dumps(validation_record,indent=2))


## Scientific and operational cautions

- The smoke test is a pipeline check, not a statistically powered comparison.
- Do not pool ToolRet and MCPToolBench++ metrics.
- Do not tune IDF, preprocessing, or thresholds on evaluator qrels.
- The model hash, dataset manifests, query/tool counts and exclusions must accompany any reported result.
- The notebook does not merge the PR or alter the original E8 run.

Official model documentation: https://fasttext.cc/docs/en/crawl-vectors
